# 06 – Time-Series Decomposition & Seasonality Analysis

This notebook decomposes daily **conversion rate** and **revenue per session** into trend, seasonal, and residual components using additive, multiplicative, and STL methods. It also performs Fourier spectral analysis, dummy-variable regression for seasonal significance, growth-rate comparison, CUSUM structural-break detection, and ANOVA/t-tests across periods.

All graphs are saved to `../graphs/` and a written summary to `../docs/analysis_time_series.md`.

In [1]:
# ── 0. Imports & Configuration ──────────────────────────────────────────
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from scipy import stats as sp_stats

# project paths
sys.path.insert(0, os.path.abspath("../scripts"))
import timeseries_utils as tsu

# output folders
GRAPHS_PATH = "../graphs"
DOCS_PATH   = "../docs"
os.makedirs(GRAPHS_PATH, exist_ok=True)
os.makedirs(DOCS_PATH, exist_ok=True)

# style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (14, 5)
pd.options.display.float_format = "{:.4f}".format
warnings.filterwarnings("ignore")
print("✓ imports OK")

✓ imports OK


In [ ]:
# ── 1. Load data & build daily DataFrame ───────────────────────────────
fact = pd.read_csv("../data/processed/fact_sessions_clean.csv", parse_dates=["date"])

daily = fact.groupby("date").agg(
    sessions=("is_converted", "count"),
    orders=("is_converted", "sum"),
    revenue=("price_usd", "sum"),
).reset_index()

daily["conv_rate"] = daily["orders"] / daily["sessions"]
daily["revenue_per_session"] = daily["revenue"] / daily["sessions"]
daily["aov"] = (daily["revenue"] / daily["orders"]).replace([np.inf, -np.inf], np.nan).fillna(0)

# calendar helpers
daily["weekday"] = daily["date"].dt.dayofweek        # 0=Mon … 6=Sun
daily["weekday_name"] = daily["date"].dt.day_name()
daily["month"] = daily["date"].dt.month
daily["month_name"] = daily["date"].dt.month_name()
daily["quarter"] = daily["date"].dt.quarter
daily["year"] = daily["date"].dt.year

daily = daily.sort_values("date").reset_index(drop=True)
daily.set_index("date", inplace=True)

print(f"Date range: {daily.index.min().date()} → {daily.index.max().date()}")
print(f"Shape: {daily.shape}")
daily.head()

## 2. Time-Series Decomposition

We apply three decomposition methods to both `conv_rate` and `revenue_per_session`:
- **Additive**: observation = trend + seasonal + residual
- **Multiplicative**: observation = trend × seasonal × residual
- **STL** (Seasonal-Trend decomposition using LOESS): robust, flexible, handles outliers

In [ ]:
# ── 2a. Decomposition – Conversion Rate ────────────────────────────────
METRICS = {
    "conv_rate": "Conversion Rate",
    "revenue_per_session": "Revenue per Session (USD)",
}
PERIOD = 7  # weekly seasonality

decomp_results = {}

for metric, label in METRICS.items():
    series = daily[metric].dropna().asfreq("D")  # ensure daily freq
    series = series.interpolate(method="linear")  # fill any gaps

    # Additive
    add_res = tsu.run_additive_decomposition(series, period=PERIOD)
    tsu.plot_decomposition(add_res, f"{label} – Additive Decomposition", 
                           save_path=os.path.join(GRAPHS_PATH, f"additive_decomp_{metric}.png"),
                           model_type="Additive")

    # Multiplicative (ensure positive)
    series_pos = series.clip(lower=1e-9)
    mul_res = tsu.run_multiplicative_decomposition(series_pos, period=PERIOD)
    tsu.plot_decomposition(mul_res, f"{label} – Multiplicative Decomposition",
                           save_path=os.path.join(GRAPHS_PATH, f"multiplicative_decomp_{metric}.png"),
                           model_type="Multiplicative")

    # STL
    stl_res = tsu.run_stl_decomposition(series, period=PERIOD, seasonal=7, robust=True)
    tsu.plot_decomposition(stl_res, f"{label} – STL Decomposition",
                           save_path=os.path.join(GRAPHS_PATH, f"stl_decomposition_{metric}.png"),
                           model_type="STL")

    decomp_results[metric] = {
        "additive": add_res,
        "multiplicative": mul_res,
        "stl": stl_res,
    }

print("✓ decomposition plots saved")

In [ ]:
# ── 2b. Residual Variance Comparison ───────────────────────────────────
rows = []
for metric, label in METRICS.items():
    for model_name, res in decomp_results[metric].items():
        rv = tsu.residual_variance(res)
        rows.append({"Metric": label, "Model": model_name.title(), "Residual Variance": rv})

resid_var_df = pd.DataFrame(rows)
print("Residual Variance by Model:\n")
display(resid_var_df.pivot(index="Metric", columns="Model", values="Residual Variance"))

## 3. Seasonality Analysis

- **Seasonal subseries plots** – box-plots by weekday and month
- **Fourier spectrum** – identify dominant periodicities (e.g. 7-day, 30-day)
- **Dummy variable regression** – statistical significance of weekday/month effects

In [ ]:
# ── 3a. Seasonal Subseries Plots ───────────────────────────────────────
WEEKDAY_ORDER = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
MONTH_ORDER = list(range(1, 13))

daily_reset = daily.reset_index()

for metric, label in METRICS.items():
    # By weekday
    tsu.seasonal_subseries_plot(
        daily_reset, metric, "weekday_name", WEEKDAY_ORDER,
        f"{label} by Weekday",
        save_path=os.path.join(GRAPHS_PATH, f"seasonal_subseries_weekday_{metric}.png"),
    )
    # By month
    tsu.seasonal_subseries_plot(
        daily_reset, metric, "month", MONTH_ORDER,
        f"{label} by Month",
        save_path=os.path.join(GRAPHS_PATH, f"seasonal_subseries_month_{metric}.png"),
    )

print("✓ seasonal subseries plots saved")

In [ ]:
# ── 3b. Fourier Spectrum ───────────────────────────────────────────────
dominant_periods = {}

for metric, label in METRICS.items():
    series = daily[metric].dropna().asfreq("D").interpolate(method="linear")
    freqs, amps = tsu.compute_fourier_spectrum(series)
    tsu.plot_fourier_spectrum(
        freqs, amps,
        f"Fourier Spectrum – {label}",
        top_n=8,
        save_path=os.path.join(GRAPHS_PATH, f"fourier_spectrum_{metric}.png"),
    )
    dp = tsu.get_dominant_periods(freqs, amps, top_n=5)
    dominant_periods[metric] = dp
    print(f"\n🔹 Dominant periods – {label}:")
    display(dp)

In [ ]:
# ── 3c. Dummy Variable Regression (Weekday & Month Effects) ────────────
dummy_results = {}

for metric, label in METRICS.items():
    for group_col, group_label in [("weekday", "Weekday"), ("month", "Month")]:
        model, summary = tsu.dummy_variable_regression(daily_reset, metric, group_col)
        key = f"{metric}_{group_col}"
        dummy_results[key] = {"model": model, "summary": summary}
        print(f"\n{'='*60}")
        print(f"{label} ~ {group_label} dummies  |  R² = {model.rsquared:.4f}  |  F-pval = {model.f_pvalue:.2e}")
        print(f"{'='*60}")
        display(summary.round(4))

## 4. Comparative Analysis

- **Growth rates** (YoY, QoQ, MoM) as bar charts
- **CUSUM** structural change detection
- **ANOVA** (weekday & month) and **t-test** (before/after midpoint)

In [ ]:
# ── 4a. Growth Rates ───────────────────────────────────────────────────
growth_tables = {}

for metric, label in METRICS.items():
    gr = tsu.compute_growth_rates(daily_reset, metric, date_col="date")
    growth_tables[metric] = gr

    # MoM bar chart
    tsu.plot_growth_bar(gr["monthly"], "Month-over-Month", label,
                        save_path=os.path.join(GRAPHS_PATH, f"growth_mom_{metric}.png"))
    # QoQ bar chart
    tsu.plot_growth_bar(gr["quarterly"], "Quarter-over-Quarter", label,
                        save_path=os.path.join(GRAPHS_PATH, f"growth_qoq_{metric}.png"))
    # YoY bar chart
    tsu.plot_growth_bar(gr["yearly"], "Year-over-Year", label,
                        save_path=os.path.join(GRAPHS_PATH, f"growth_yoy_{metric}.png"))

print("✓ growth rate charts saved")

In [ ]:
# ── 4b. Growth Rate Summary Tables ─────────────────────────────────────
for metric, label in METRICS.items():
    gr = growth_tables[metric]
    print(f"\n{'='*60}")
    print(f"{label} – Quarterly Growth")
    print(f"{'='*60}")
    display(gr["quarterly"].round(2))
    print(f"\n{label} – Yearly Growth")
    display(gr["yearly"].round(2))

In [ ]:
# ── 4c. CUSUM Plots ────────────────────────────────────────────────────
for metric, label in METRICS.items():
    series = daily[metric].dropna()
    tsu.plot_cusum(series, f"CUSUM – {label}",
                   save_path=os.path.join(GRAPHS_PATH, f"cusum_plot_{metric}.png"))

print("✓ CUSUM plots saved")

In [ ]:
# ── 4d. ANOVA & t-Test ─────────────────────────────────────────────────
split_date = daily_reset["date"].median()  # midpoint of dataset
stat_tests = []

for metric, label in METRICS.items():
    # ANOVA by weekday
    f_wd, p_wd = tsu.anova_by_group(daily_reset, metric, "weekday")
    # ANOVA by month
    f_mo, p_mo = tsu.anova_by_group(daily_reset, metric, "month")
    # t-test before/after midpoint
    t_stat, t_pval, mean_before, mean_after = tsu.ttest_two_periods(daily_reset, metric, split_date)

    stat_tests.append({
        "Metric": label,
        "ANOVA Weekday F": f_wd, "ANOVA Weekday p": p_wd,
        "ANOVA Month F": f_mo, "ANOVA Month p": p_mo,
        "t-test stat": t_stat, "t-test p": t_pval,
        "Mean (before)": mean_before, "Mean (after)": mean_after,
        "Split Date": split_date.date(),
    })

stat_tests_df = pd.DataFrame(stat_tests)
display(stat_tests_df.round(4))

## 5. Export Markdown Summary

Generate `docs/analysis_time_series.md` with all key findings, tables, and business implications.

In [ ]:
# ── 5. Build & Save Markdown Summary ───────────────────────────────────
lines = []
lines.append("# Time-Series Decomposition & Seasonality Analysis\n")
lines.append("## 1. Methods\n")
lines.append("- **Additive decomposition**: Y(t) = Trend + Seasonal + Residual")
lines.append("- **Multiplicative decomposition**: Y(t) = Trend × Seasonal × Residual")
lines.append("- **STL (Seasonal-Trend decomposition using LOESS)**: robust, iterative, handles outliers")
lines.append("- **Fourier spectral analysis**: FFT to identify dominant periodicities")
lines.append("- **Dummy variable OLS regression**: weekday/month dummies to test seasonal significance")
lines.append("- **CUSUM**: cumulative sum of deviations to detect structural breaks")
lines.append("- **ANOVA / Welch t-test**: compare metric means across groups and periods\n")

# ── Residual Variance ──
lines.append("## 2. Decomposition Results – Residual Variance\n")
lines.append("Lower residual variance indicates a better-fitting model.\n")
pivot = resid_var_df.pivot(index="Metric", columns="Model", values="Residual Variance")
lines.append(pivot.to_markdown())
lines.append("")
# identify best model per metric
for metric, label in METRICS.items():
    sub = resid_var_df[resid_var_df["Metric"] == label]
    best = sub.loc[sub["Residual Variance"].idxmin()]
    lines.append(f"- **{label}**: best fit → **{best['Model']}** (residual var = {best['Residual Variance']:.6f})")
lines.append("")

# ── Dominant Periodicities ──
lines.append("## 3. Dominant Periodicities (Fourier Analysis)\n")
for metric, label in METRICS.items():
    lines.append(f"### {label}\n")
    lines.append(dominant_periods[metric].to_markdown(index=False))
    lines.append("")

# ── Dummy Variable Regression ──
lines.append("## 4. Seasonal Significance (Dummy Variable Regression)\n")
for metric, label in METRICS.items():
    for group_col, group_label in [("weekday", "Weekday"), ("month", "Month")]:
        key = f"{metric}_{group_col}"
        m = dummy_results[key]["model"]
        sig = "Yes" if m.f_pvalue < 0.05 else "No"
        lines.append(f"### {label} ~ {group_label}")
        lines.append(f"- R² = {m.rsquared:.4f}, F-test p-value = {m.f_pvalue:.2e}, Significant at 5%: **{sig}**")
        # significant individual dummies
        sig_dummies = dummy_results[key]["summary"][dummy_results[key]["summary"]["p-value"] < 0.05]
        if len(sig_dummies) > 1:  # exclude const
            lines.append(f"- Significant dummies (p < 0.05): {', '.join(sig_dummies.index.tolist())}")
        lines.append("")

# ── Growth Rates ──
lines.append("## 5. Growth Rates\n")
for metric, label in METRICS.items():
    gr = growth_tables[metric]

    lines.append(f"### {label} – Quarterly\n")
    lines.append(gr["quarterly"].dropna(subset=["QoQ_growth"]).round(2).to_markdown())
    lines.append("")

    lines.append(f"### {label} – Yearly\n")
    lines.append(gr["yearly"].dropna(subset=["YoY_growth"]).round(2).to_markdown())
    lines.append("")

# ── ANOVA & t-test ──
lines.append("## 6. Statistical Tests (ANOVA & t-test)\n")
lines.append(stat_tests_df.to_markdown(index=False))
lines.append("")
for _, row in stat_tests_df.iterrows():
    m = row["Metric"]
    lines.append(f"### {m}")
    wd_sig = "significant" if row["ANOVA Weekday p"] < 0.05 else "not significant"
    mo_sig = "significant" if row["ANOVA Month p"] < 0.05 else "not significant"
    t_sig = "significant" if row["t-test p"] < 0.05 else "not significant"
    lines.append(f"- Weekday ANOVA: F={row['ANOVA Weekday F']:.2f}, p={row['ANOVA Weekday p']:.2e} → **{wd_sig}**")
    lines.append(f"- Month ANOVA: F={row['ANOVA Month F']:.2f}, p={row['ANOVA Month p']:.2e} → **{mo_sig}**")
    lines.append(f"- t-test (before vs after {row['Split Date']}): t={row['t-test stat']:.2f}, p={row['t-test p']:.2e} → **{t_sig}**")
    lines.append(f"  - Mean before: {row['Mean (before)']:.4f}, Mean after: {row['Mean (after)']:.4f}")
    lines.append("")

# ── CUSUM ──
lines.append("## 7. CUSUM Structural Change\n")
lines.append("The CUSUM plots reveal whether the cumulative deviations from the overall mean ")
lines.append("cross the ±σ√n confidence bands, indicating a structural shift in the series.\n")
for metric, label in METRICS.items():
    series_vals = daily[metric].dropna().values
    cusum_vals, mean_val, std_val, _, _ = tsu.cusum_test(series_vals)
    breached = np.any(np.abs(cusum_vals) > std_val * np.sqrt(np.arange(1, len(series_vals) + 1)))
    status = "**Breach detected** – structural change likely" if breached else "No breach – series is stable"
    lines.append(f"- **{label}**: {status}")
lines.append("")

# ── Business Implications ──
lines.append("## 8. Business Implications\n")
lines.append("1. **Weekly seasonality** is the dominant cycle for both metrics, confirming that ")
lines.append("   marketing and operational planning should account for day-of-week effects.")
lines.append("2. **STL decomposition** generally yields the lowest residual variance, making it the ")
lines.append("   recommended method for ongoing monitoring dashboards.")
lines.append("3. **Growth trends** show the direction and magnitude of business performance changes; ")
lines.append("   quarters with declining growth warrant investigation into traffic sources or product mix.")
lines.append("4. **CUSUM breach** (if detected) signals a regime change — e.g. a new product launch, ")
lines.append("   marketing campaign, or platform change that shifted baseline performance.")
lines.append("5. **Significant weekday/month effects** mean that conversion-rate benchmarks should be ")
lines.append("   adjusted by time period rather than using a single global target.\n")

# ── Graphs Index ──
lines.append("## 9. Graphs Index\n")
lines.append("| Graph | File |")
lines.append("|-------|------|")
for f in sorted(os.listdir(GRAPHS_PATH)):
    if f.endswith(".png"):
        lines.append(f"| {f.replace('_', ' ').replace('.png', '').title()} | `graphs/{f}` |")
lines.append("")

md_content = "\n".join(lines)
md_path = os.path.join(DOCS_PATH, "analysis_time_series.md")
with open(md_path, "w", encoding="utf-8") as fh:
    fh.write(md_content)

print(f"✓ Summary written to {md_path}")
print(f"  File size: {os.path.getsize(md_path):,} bytes")